***Protein Protein Interaction Project***

*Authors*: Jordan Harrison, Haoyue Hu

In [17]:
# Import necessary Libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit

In [2]:
# Load the dataset
loaded = False


"""
have this in just as a way to check if the files are in the right place. the github is structured to have two data
folders (don't ask me why I don't know), but the when we submit the files could be in the same dir
"""
try:
    int_data = pd.read_csv("Data/Data/Supp-AB.tsv", sep='\t')
    seq_data = pd.read_csv("Data/Data/SEQ-Supp-ABCD.tsv", sep='\t', header = None, names = ['id', 'seq'])
    seq_map = dict(zip(seq_data.id, seq_data.seq))
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print("Checking local directory for files...")
try:
    int_data = pd.read_csv("Supp-AB.tsv", sep='\t')
    seq_data = pd.read_csv("SEQ-Supp-ABCD.tsv", sep='\t', header = None)
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print("Files not found in local directory")

# Check everything loaded correctly
pair_ids = set(int_data.v1) | set(int_data.v2)
print(f"Number of interacting protein pairs: {len(int_data)}")
print(f"Number of protein sequences: {len(seq_data)}")
print(f"Number of unique protein pairs: {len(pair_ids)}")

# Check data set
print(int_data.head())
#int_data.info()
print(seq_data.head())
#seq_data.info()

Data loaded successfully.
Files not found in local directory
Number of interacting protein pairs: 73108
Number of protein sequences: 10370
Number of unique protein pairs: 10364
               v1              v2  label
0     NP_663777.1     NP_001233.1      1
1     NP_055436.2     NP_001420.2      1
2     NP_003630.1  NP_001073594.1      1
3  NP_001001998.1     NP_067000.1      1
4     NP_001888.2     NP_001782.1      1
       id                                                seq
0  A0AVK6  MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...
1  A2A3N6  MASEVPYASGMPIKKIGHRSVDSSGGTTSSALKGAIQLGITHTVGS...
2  A2A3N6  MASEVPYASGMPIKKIGHRSVDSSGGTTSSALKGAIQLGITHTVGS...
3  A6H8Y1  MFRRARLSVKPNVRPGVGARGSTASNPQRGRESPRPPEPATDSASK...
4  A6H8Y1  MFRRARLSVKPNVRPGVGARGSTASNPQRGRESPRPPEPATDSASK...


In [3]:
# create new dataframe with the protein sequences and their interactions
df = int_data.copy()
df['protein_A_seq'] = df.v1.map(seq_map)
df['protein_B_seq'] = df.v2.map(seq_map)

"""
To avoid redunancy, the pair AB and BA are the same thing so we need to check for any duplicates
"""
# wizardry here, creates columns 'a' and 'b' then uses the min and max (which works alphabetically) to order the columns
# if you get ab and the ba this function puts ab into both of the new columns B)
df['a'] = df[['v1', 'v2']].min(axis=1)
df['b'] = df[['v1', 'v2']].max(axis=1)
df = df.drop_duplicates(subset=['a','b','label']).reset_index(drop=True)

# clean it up, apply good labels, see the balance
df = df.rename(columns={'v1':'id_A','v2':'id_B'})[['id_A','id_B','protein_A_seq','protein_B_seq','label']]
print(df.label.value_counts(), df.shape)
df.head()



label
1    36629
0    36479
Name: count, dtype: int64 (73108, 5)


,id_A,id_B,protein_A_seq,protein_B_seq,label
0,NP_663777.1,NP_001233.1,MESSKKMDSPGALQTNPPLKLHTDRSAGTPVFVPEQGGYKEKFVKT...,MARPHPWWLCVLGTLVGLSATPAPKSCPERHYWAQGKLCCQMCEPG...,1
1,NP_055436.2,NP_001420.2,MVMSSYMVNSKYVDPKFPPCEEYLQGGYLGEQGADYYGGGAQGADF...,MAENVVEPGPPSAKRPKLSSPALSASASDGTDFGSLFDLEHDLPDE...,1
2,NP_003630.1,NP_001073594.1,MNRHLWKSQLCEMVQPSGGPAADQDVLGEESPLGKPAMLHLPSEQG...,MEGGRRARVVIESKRNFFLGAFPTPFPAEHVELGRLGDSETAMVPG...,1
3,NP_001001998.1,NP_067000.1,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,MLFYSFFKSLVGKDVVVELKNDLSICGTLHSVDQYLNIKLTDISVT...,1
4,NP_001888.2,NP_001782.1,MQSGPRPPLPAPGLALALTLTMLARLASAASFFGENHLEVPVATAL...,MQTIKCVVVGDGAVGKTCLLISYTTNKFPSEYVPTVFDNYAVTVMI...,1


In [ ]:
# Setup splits

"""
I want to try two things here:
1.) A random split, just pick random pairs for the train, validation, and test set
2.) Split based on proteins, ie randomly split the data set so it sees proteins that it has never seen in the test/validation sets
    this should in theory prevent the model from memorizing the "sequences"
"""
data = df.to_numpy()
X = data[:,2:4]
y = data[:,4]
labels = data[:,0:2]

# Option 1, random split

X_train, X_test, y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Option 2, split via proteins and then form sets

# Sort the proteins by labels
proteins = np.array(sorted(set(df.id_A) | set(df.id_B)))

#Initialize Group Shuffle Split
gss = GroupShuffleSplit(n_splits = 1, test_size = 0.3, random_state = 42)

#Use the set function to drop duplicates
gss_idx_train, gss_idx_test = next(gss.split(proteins, groups = proteins))
gss_train_prots, gss_test_prots = set(proteins[gss_idx_train]), set(proteins[gss_idx_test])

#check if the proteins are in the set
a_train, b_train = df.id_A.isin(gss_train_prots), df.id_B.isin(gss_train_prots)

#Keep unique entrys (both True, or both False) to form the training and test set
gss_X_train = df[a_train & b_train]
gss_X_test = df[~a_train & ~b_train]

print(len(gss_X_train), len(gss_X_test), (a_train ^ b_train).sum())


36251 6391 30466


In [ ]:
# Featurize this sucker